# 06. 전체 데이터 라벨 수정 + 학습 (A100, 약 3~4시간, 한 번에 끝까지)

01(Val 판정) → 02(Train 판정) → 03(라벨 수정·삭제) → 04(학습·점수표)를 노트북 하나로 이어서 돌린다.
빨리 시험만 해 보려면 0단계의 `TRAIN_FRAC`을 0.1로 바꾼다(Train 10%, 약 40~60분). Val은 항상 전체를 쓴다.

**돌리는 사람에게**
1. 공유받은 `증상인식_최종_프로젝트` 폴더를 '내 드라이브'에 **바로가기 추가**(폴더 우클릭 → 정리 → 바로가기 추가 → 내 드라이브).
   다른 곳에 두었으면 0단계의 `PROJECT_ROOT`만 바꾼다.
2. 런타임 → 런타임 유형 변경 → **A100**.
3. 맨 위 셀(pip)을 실행하고, '세션 다시 시작' 메시지가 나오면 다시 시작한 뒤 **두 번째 셀부터** 끝까지 실행(런타임 → 이후 셀 모두 실행).
4. 끊기면 그냥 다시 실행: Qwen 판정과 학습은 저장된 만큼 건너뛰고 이어서 한다.

| 단계 | 하는 일 | 시간(A100) |
|---|---|---|
| 0 | 준비 | 1분 |
| 1 | Qwen으로 Val 전체 판정 | 15~30분 |
| 2 | Qwen으로 Train 판정 | 1~2시간 |
| 3 | 라벨 수정·애매한 칸 삭제, 검수 파일 | 1분 |
| 4 | KoELECTRA 학습: 정리 모델 + 원본 모델(비교용) | 각 약 40분 |
| 5 | 점수표 + 실험기록 + 결과 모으기 | 1분 |

결과: **`results/06_결과/` 한 폴더**(점수표, 처리 분포표, 검수 파일, 판정 표, 고친 라벨) + `models/정리모델.pt`, `models/원본모델.pt`(시연 05가 씀).
시간이 모자라면 4단계의 `TRAIN_ORIGINAL`을 False로 두면 원본 모델 학습(약 40분)을 건너뛴다.
원본 JSON과 `labels_df.pkl`은 읽기만 한다.

검수(01 다음 Claude·사람 확인)는 건너뛰고 모든 증상을 '수정' 규칙대로 처리한다.
검수 파일(`검수_val.xlsx`, `검수_val.txt`)은 만들어 두므로 나중에 확인해서, 못 믿을 증상이 있으면 3단계의 `MODES`만 바꿔 3단계부터 다시 돌리면 된다.

## 0. 준비

In [ ]:
# Qwen3는 transformers 4.51 이상이 필요하다. 설치 후 '세션 다시 시작' 메시지가 나오면 다시 시작하고 다음 셀부터 실행
!pip -q install -U "transformers>=4.51" accelerate

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from src import config
from src.config import TrainConfig
from src.data import load_labels, prep, symptom_matrix
from src.cells import build_cells, judge_cells, save_judged
from src.cleaning import decide_actions, action_summary, build_fixed, save_fixed, review_sample, write_review_txt
from src.judge_llm import load_judge
from src.model import encode_chunks
from src.trainer import seed_all, train_model, predict
from src.infer import save_model
from src.evaluate import claude_fixed_val, score_table, agree_with_claude, log_experiment
from src.metrics import pr_report

jcfg = config.JudgeConfig()
cfg = TrainConfig()
TRAIN_FRAC = 1.0                          # Train 중 쓸 비율(1.0 = 전체). 시험만 하려면 0.1. Val은 항상 전체
TAG = f"{round(TRAIN_FRAC * 100)}pct"
SFX = "" if TRAIN_FRAC >= 1 else f"_{TAG}"   # 일부만 쓰면 파일 이름에 붙여 전체 결과와 섞이지 않게 함
OUT = f"{config.RESULTS_DIR}/06_결과{SFX}"   # 결과를 모으는 폴더
for d in [OUT, config.DATA_DIR, config.MODEL_DIR, config.JUDGE_DIR, f"{config.RESULTS_DIR}/04_final"]:
    os.makedirs(d, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(GPU 런타임으로 바꾸기)")

In [ ]:
# TRAIN_FRAC < 1이면 Train 일부 고르기. 고른 대화 목록을 파일로 남겨 두고, 다시 돌릴 때(04 포함)는 같은 목록을 쓴다
tr_df, va_df = load_labels()
if TRAIN_FRAC < 1:
    SUB_PATH = f"{config.DATA_DIR}/train_{TAG}_files.csv"
    if os.path.exists(SUB_PATH):
        keep = set(pd.read_csv(SUB_PATH)["file_name"])
        print("저장된 목록 사용:", SUB_PATH)
    else:
        keep = set(tr_df.sample(frac=TRAIN_FRAC, random_state=cfg.seed)["file_name"])
        pd.DataFrame({"file_name": sorted(keep)}).to_csv(SUB_PATH, index=False)
        print("새 목록 저장:", SUB_PATH)
    tr_df = tr_df[tr_df["file_name"].isin(keep)]
print(f"Train {len(tr_df):,}개 대화 ({TRAIN_FRAC:.0%}) / Val {len(va_df):,}개 대화(전체)")

## 1. Qwen으로 Val 판정
블록(2,000칸)마다 드라이브에 저장한다. 끊기면 이 셀부터 다시 실행하면 이어서 한다.

In [ ]:
# 처음엔 모델 내려받기에 몇 분 걸린다(약 16GB)
tok_q, llm = load_judge(jcfg.model_name, jcfg.load_in_4bit)
va_cells = build_cells(va_df)
va_probs, done = judge_cells(va_cells, tok_q, llm, f"{config.JUDGE_DIR}/val", jcfg)
assert done, "Val 판정이 덜 끝남. 이 셀을 다시 실행"

## 2. Qwen으로 Train 판정
가장 오래 걸리는 단계(1~2시간). 끊기면 이 셀부터 다시 실행하면 이어서 한다.

In [ ]:
tr_cells = build_cells(tr_df)
tr_probs, done = judge_cells(tr_cells, tok_q, llm, f"{config.JUDGE_DIR}/train{SFX}", jcfg)
assert done, "Train 판정이 덜 끝남. 이 셀을 다시 실행"
del llm; torch.cuda.empty_cache()   # 학습 전에 GPU 메모리 비우기

## 3. 라벨 수정·애매한 칸 삭제 + 검수 파일
`MODES`: 증상마다 "수정"(기본) / "삭제만"(수정할 칸도 삭제로) / "안 함"(원래 라벨 그대로).
'경고'가 뜬 증상은 정리 후 '있음'이 너무 적다는 뜻이다(Train 일부만 쓰면 기준도 그 비율만큼 줄임).

In [ ]:
MODES = {s: "수정" for s in config.LABELS}
# 예: MODES["전신쇠약"] = "삭제만"
va_j = decide_actions(save_judged(va_cells, va_probs, f"{config.JUDGE_DIR}/val_judged.csv"), jcfg, MODES)
tr_j = decide_actions(save_judged(tr_cells, tr_probs, f"{config.JUDGE_DIR}/train{SFX}_judged.csv"), jcfg, MODES)
s_tr = action_summary(tr_j, tr_df, round(jcfg.warn_min_pos_train * TRAIN_FRAC), jcfg)
s_va = action_summary(va_j, va_df, jcfg.warn_min_pos_val, jcfg)
pd.concat({"Train": s_tr, "Val": s_va}, axis=1).to_csv(f"{OUT}/처리_분포표.csv", encoding="utf-8-sig")
print(f"[Train {TRAIN_FRAC:.0%}]"); display(s_tr)
print("[Val]"); display(s_va)

agree = agree_with_claude(va_j)   # 대회 때 Claude 판정과 일치율(파일 있을 때만)
if agree is not None:
    agree.to_csv(f"{OUT}/Claude_일치율.csv", encoding="utf-8-sig")
    display(agree)

sample = review_sample(va_j)
sample.to_excel(f"{OUT}/검수_val.xlsx", index=False)
write_review_txt(sample, dict(zip(va_cells["대화ID"], va_cells["대화"])), f"{OUT}/검수_val.txt")

In [ ]:
# 고친 라벨 + 삭제 마스크 저장 (04와 같은 행 순서)
tok = AutoTokenizer.from_pretrained(cfg.plm_name)
tr = prep(tr_df, tok.sep_token, drop_empty=True)
va = prep(va_df, tok.sep_token, drop_empty=False)
print("Train"); Y_tr, M_tr = build_fixed(tr, tr_j)
print("Val");   Y_va, M_va = build_fixed(va, va_j)
save_fixed(f"{config.DATA_DIR}/train_fixed{SFX}.npz", tr, Y_tr, M_tr)
save_fixed(f"{config.DATA_DIR}/val_fixed.npz", va, Y_va, M_va)
Y_tr0, Y_va0 = symptom_matrix(tr["symptom"]), symptom_matrix(va["symptom"])   # 원래 라벨
print(pd.DataFrame({"Train 원래 있음": Y_tr0.sum(0), "Train 최종 있음": (Y_tr * M_tr).sum(0),
                    "Val 원래 있음": Y_va0.sum(0), "Val 최종 있음": (Y_va * M_va).sum(0)},
                   index=config.LABELS).astype(int).to_string())

## 4. KoELECTRA 학습 → 모델 저장 → Val 확률 저장
정리 모델(수정 라벨)과 원본 모델(원래 라벨)을 같은 Train으로 학습해 비교한다. 끝난 모델은 다시 실행해도 건너뛴다.

In [ ]:
ch_tr = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(tr["text"], desc="Train")]
ch_va = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(va["text"], desc="Val")]

def run(name, Y, M):
    prob_path = f"{OUT}/{name}_val_prob.npy"
    if os.path.exists(prob_path):
        print(name, "이미 끝남 → 건너뜀")
        return np.load(prob_path)
    seed_all(cfg.seed)
    model = train_model(ch_tr, Y, cfg, tok, M, desc=name, loss_csv=f"{OUT}/{name}_loss.csv")
    save_model(model, f"{config.MODEL_DIR}/{name}.pt", config.LABELS, cfg)
    p = predict(model, ch_va, tok, desc=f"{name} Val")
    np.save(prob_path, p)
    del model; torch.cuda.empty_cache()
    return p

TRAIN_ORIGINAL = True      # 원본 모델(비교용)도 학습할지. 시간이 없으면 False
probs = {"정리 모델": run(f"정리모델{SFX}", Y_tr, M_tr)}
if TRAIN_ORIGINAL:
    probs["원본 모델"] = run(f"원본모델{SFX}", Y_tr0, None)

## 5. 점수표 (Macro F1, 기준값 0.5) + 실험기록

In [ ]:
val_sets = {"정리 Val": (Y_va, M_va), "원본 Val": (Y_va0, None)}
fixed = claude_fixed_val(va_df)
if fixed is not None:
    val_sets["고친 Val"] = fixed
macro, detail = score_table(probs, val_sets)
macro.to_csv(f"{OUT}/점수_macro.csv", encoding="utf-8-sig")
detail.to_csv(f"{OUT}/점수_증상별.csv", encoding="utf-8-sig")
display(macro)
display(detail)

pr = pr_report(Y_va, probs["정리 모델"], config.LABELS, 0.5, M_va)   # 정리 모델, 정리 Val
pr.to_csv(f"{OUT}/재현율_정밀도.csv", encoding="utf-8-sig")
display(pr)

In [ ]:
part = "" if TRAIN_FRAC >= 1 else f", Train {TRAIN_FRAC:.0%}"
SETTINGS = {"정리 모델": f"Qwen 수정 라벨 + 애매한 칸 삭제{part}, 기본 설정",
            "원본 모델": f"원래 라벨{part}, 기본 설정"}
for m in probs:
    det = {}
    for v, (Y, M) in val_sets.items():
        t = pr_report(Y, probs[m], config.LABELS, 0.5, M)
        t["F1"] = detail[f"{m} / {v}"].reindex(t.index)
        det[v] = t
    log = log_experiment(f"{config.RESULTS_DIR}/04_final/실험기록.csv", m + ("" if TRAIN_FRAC >= 1 else f" (Train {TAG})"),
                         SETTINGS[m], macro.loc[m].to_dict(), det)
display(log[[c for c in log.columns if "Macro" in c or c in ("실험", "바꾼 설정")]])

## 6. 결과 모으기
판정 표와 고친 라벨을 결과 폴더로 복사한다. 이 폴더 하나만 보면 된다(모델은 크기가 커서 `models/`에 그대로 둔다).

In [ ]:
import shutil
for p in [f"{config.JUDGE_DIR}/val_judged.csv", f"{config.JUDGE_DIR}/train{SFX}_judged.csv",
          f"{config.DATA_DIR}/train_fixed{SFX}.npz", f"{config.DATA_DIR}/val_fixed.npz",
          f"{config.RESULTS_DIR}/04_final/실험기록.csv"]:
    if os.path.exists(p):
        shutil.copy(p, OUT)
print("결과 폴더:", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(f'{OUT}/{f}') / 1e6:.1f}MB)")
print("모델:", *[f"{config.MODEL_DIR}/{n}{SFX}.pt" for n in ["정리모델", "원본모델"]], sep="\n  ")